# Per-bin closure of the `spline_cross_quad` response

PROfit's z-expansion response used to be the product $\prod_i s_i(\eta_i)$ of one-dimensional
splines. The true per-bin response is an exact multivariate quadratic in the standardized PCA
coordinates $\eta$ *with* cross terms $\eta_i\eta_j$ (see `../../notes/spline_factorization.md`
and notebook 12), so the production XMLs now carry one `type="spline_cross_quad"` systematic per
prior and `PROsyst::GetSplineFactor` evaluates

$$R(\eta) = 1 + \sum_i\left[s_i(\eta_i) - 1\right] + \sum_{i<j} e_{ij}\,\eta_i\eta_j .$$

This notebook is the **per-bin closure test** of that response. It asks, bin by bin and for every
prior, how far PROfit's reconstruction from the *stored* branches is from the exact per-event
reweighting. Three responses are compared on the same events, binned exactly as PROfit bins them:

| | response | how it is obtained here |
|---|---|---|
| **exact** | $\sum_{\rm ev} w\,\omega(\eta)\,/\,\sum_{\rm ev} w\,\omega(0)$ | the per-event quadratic-in-$F_A$ weight $\omega(\eta)$ from the production code (`zexp_reweighting`), evaluated at arbitrary $\eta$ |
| **additive** (PROfit now) | $1 + \sum_i(s_i - 1) + \sum_{i<j} e_{ij}\eta_i\eta_j$ | `PROsyst::FillSpline`, `FillSplineCrossQuad` and `GetSplineFactor` re-implemented on the stored seven-knot and CROSS branches |
| **product** (PROfit before) | $\prod_i s_i(\eta_i)$ | the same member splines multiplied |

**Closure** is the largest relative per-bin difference between *additive* and *exact*. The stored
branches are float32, so the floor is a few $10^{-7}$ per event; the pass thresholds below leave
a generous margin above that. *product* is reported alongside to show the size of the effect the
cross branches remove. The reimplementation follows the C++ line by line (segment layout,
force_0_cv normalisation, the $w>30$ / non-finite universe-weight guard of `PROcreate`, the
$e_{ij} = R(e_i+e_j) - s_i(1) - s_j(1) + 1$ extraction, and the member order of the XML `splines=`
attribute), so a change in any of those conventions shows up here as a closure failure.

Two further checks ride along: the stored knot and CROSS branches are recomputed from
`GTruth_gQ2` and `MaCCQE_UBGenie` through the production code (catches a stale file, e.g. the
8 September dipole-$F_A(0)$ mismatch), and the on-axis quadraticity statistic that
`FillSplineCrossQuad` logs at run time is reproduced.

This is the bin-level companion of notebook 14, which closes the same correction at the
posterior level (reweighted chain versus `cross/` fit).

In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

import awkward as ak
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

# Locate ma_zexp/python/scripts (shared style, fit specs) and uboone_ngem/src (the weight
# code that wrote the branches) whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
ngem_src = next(
    (parent / 'uboone_ngem' / 'src' for parent in (start, *start.parents)
     if (parent / 'uboone_ngem' / 'src' / 'zexp_reweighting.py').is_file()),
    None,
)
if helper_dir is None or ngem_src is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts and uboone_ngem/src')
for path in (helper_dir, ngem_src):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from postfit_physical_parameters import FIGURE_ROOT, PUBLICATION_RC, SPECS
import zexp_reweighting as zr

mpl.rcParams.update(PUBLICATION_RC)
np.set_printoptions(linewidth=160, precision=4, suppress=True)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

REPO_MA_ZEXP = helper_dir.parents[1]
XML_ROOT = REPO_MA_ZEXP / 'xml'
TABLE_DIR = REPO_MA_ZEXP / 'tables' / 'spline_factorization'
FIG_DIR = FIGURE_ROOT / 'spline_factorization'
PROFIT_SRC = Path('/nevis/riverside/share/epelaez/PROfit')
INPUT_FILE = Path('/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root')
TREE_NAME = 'tree'
XML_FAMILY = 'nuwro'          # any family: the MCFile selection, binning and systematic lines are identical

KNOTS = np.asarray(zr.ZEXP_SIGMA_VALUES, float)     # -3 ... 3, the knobvals of every member spline
KNOT0 = int(np.flatnonzero(KNOTS == 0)[0])
WEIGHT_GUARD = 30.0                                 # PROcreate resets universe weights > 30 or non-finite to 1

INNER_HALF, EXTENDED_HALF = 3.0, 7.0                # the knot range, and the extrapolation range the uniform-prior fits reach
TOL_INNER, TOL_EXTENDED = 1e-4, 5e-4                # pass thresholds on the largest relative per-bin difference
GRID_STEP = {2: 0.25, 3: 0.5, 4: 1.0}               # inner grid step by number of parameters
EXTENDED_STEP = {2: 0.5, 3: 1.0, 4: 2.0}
N_RANDOM = 2000                                     # random nodes per region, so half-integer and off-lattice eta are probed too
RNG_SEED = 20260924
SAVE_OUTPUTS = True
SAVE_DPI = 300

COLOR_ADDITIVE, COLOR_PRODUCT = '#0072B2', '#D55E00'   # Okabe-Ito, as in the other figures


def xml_configuration(path):
    # PROfit XMLs are not well-formed XML (bare '&&', several top-level elements).
    text = path.read_text()
    text = text[text.index('?>') + 2:] if text.lstrip().startswith('<?xml') else text
    text = re.sub(r'&(?!(amp|lt|gt|quot|apos|#\d+);)', '&amp;', text)
    root = ET.fromstring('<root>' + text + '</root>')
    channel = root.find('channel')
    bins2d = channel.find('bins2D')
    cross = {}
    for syst in root.iter('systematic'):
        if syst.get('type') == 'spline_cross_quad':
            cross[syst.text.strip()] = tuple(s.strip() for s in syst.get('splines').split(','))
    return (np.array(bins2d.get('edgesx').split(), float), np.array(bins2d.get('edgesy').split(), float),
            [s.get('name') for s in channel.findall('subchannel')], cross)


# Binning and subchannels from one XML; the `splines=` member order of every cross systematic
# from all XMLs of the family (PROfit pairs members in this order, the branch pairs them in
# zexp_cross_pairs order, and the two must agree).
ANALYSIS_EDGES_X, ANALYSIS_EDGES_Y, SUBCHANNELS, _ = xml_configuration(XML_ROOT / XML_FAMILY / 'minerva_k6.xml')
XML_MEMBER_ORDER = {}
for xml_path in sorted((XML_ROOT / XML_FAMILY).glob('*.xml')):
    ex, ey, sub, cross = xml_configuration(xml_path)
    assert np.array_equal(ex, ANALYSIS_EDGES_X) and np.array_equal(ey, ANALYSIS_EDGES_Y) and sub == SUBCHANNELS, xml_path
    for branch, members in cross.items():
        if branch in XML_MEMBER_ORDER and XML_MEMBER_ORDER[branch][0] != members:
            raise ValueError(f'{branch}: member order differs between {XML_MEMBER_ORDER[branch][1]} and {xml_path.name}')
        XML_MEMBER_ORDER[branch] = (members, xml_path.name)
N_X, N_Y = len(ANALYSIS_EDGES_X) - 1, len(ANALYSIS_EDGES_Y) - 1
N_BINS = N_X * N_Y
N_SUB = len(SUBCHANNELS)
N_BINS_FULL = N_SUB * N_BINS

PRIORS = list(zr.ZEXP_PRIORS)
FIT_KEY = {}   # prior name -> the fit key of SPECS that uses it, for labels
for spec in SPECS:
    if spec.prior is not None and not spec.uniform_prior and spec.variant == '':
        FIT_KEY.setdefault(spec.prior.name, spec.key)

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_FILE.is_file(), INPUT_FILE
print(f'Binning: {N_X} x {N_Y} = {N_BINS} bins x {N_SUB} subchannels -> {N_BINS_FULL} uncollapsed bins')
print(f'{len(PRIORS)} priors:', ', '.join(f"{p.name} ({FIT_KEY.get(p.name, '-')}, {len(p.free_a_values)} par)" for p in PRIORS))
print('PROfit     :', subprocess.run(['git', '-C', str(PROFIT_SRC), 'log', '-1', '--format=%h %ad %s', '--date=short'],
                                     capture_output=True, text=True).stdout.strip())
print('uboone_ngem:', subprocess.run(['git', '-C', str(ngem_src.parent), 'log', '-1', '--format=%h %ad %s', '--date=short'],
                                     capture_output=True, text=True).stdout.strip())
print('input file :', INPUT_FILE, time.strftime('%Y-%m-%d %H:%M', time.localtime(INPUT_FILE.stat().st_mtime)))

## Events, binning and base weight

The simulated overlay component of the PROfit `MCFile` selection (`weight_1`), with
`weight_2 = non_genie_net_weight` as the base weight. `weight_3` is the prior's own CV weight;
it enters PROfit's CV spectrum but not the spline universes (`include_only_weights="1,2"`), and
with `force_0_cv` every ratio is taken to the $\eta=0$ universe, so it cancels from every
response compared here. The POT scale cancels the same way. Vector branches are read in
chunks and only the selected rows kept.

In [ ]:
t0 = time.time()
SCALAR_COLUMNS = ['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
                  'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'GTruth_gQ2', 'non_genie_net_weight']
VECTOR_COLUMNS = ['MaCCQE_UBGenie', *zr.ZEXP_VARIATION_BRANCHES, *zr.ZEXP_CROSS_BRANCHES]
with uproot.open(INPUT_FILE) as root_file:
    tree = root_file[TREE_NAME]
    scalars = tree.arrays(SCALAR_COLUMNS, library='np')
    selected = ((scalars['isdata'] == 0) & (scalars['isext'] == 0) & (scalars['isdirt'] == 0)
                & (scalars['isnuwro'] == 0) & (scalars['afro_1mu1p_sel'] == 1))
    parts = {c: [] for c in VECTOR_COLUMNS}
    n_done = 0
    for chunk in tree.iterate(VECTOR_COLUMNS, step_size=200_000, library='ak'):
        mask = selected[n_done:n_done + len(chunk)]
        n_done += len(chunk)
        if not mask.any():
            continue
        sub = chunk[mask]
        for c in VECTOR_COLUMNS:
            parts[c].append(ak.to_numpy(sub[c]).astype(float))
    assert n_done == len(selected)
vectors = {c: np.concatenate(v) for c, v in parts.items()}
del parts
print(f'Read {len(selected):,} rows in {time.time() - t0:.0f} s; selected simulated events: {selected.sum():,}')

log_q2_reco = np.log10(scalars['afro_1mu1p_Q2'][selected].astype(float))
pn_reco = scalars['afro_1mu1p_Pn'][selected].astype(float)
ix = np.searchsorted(ANALYSIS_EDGES_X, log_q2_reco, side='right') - 1
iy = np.searchsorted(ANALYSIS_EDGES_Y, pn_reco, side='right') - 1
in_range = (ix >= 0) & (ix < N_X) & (iy >= 0) & (iy < N_Y)
subchannel = np.where(scalars['afro_1mu1p_true'][selected] == 1, 0, 1)

U_IDX = (subchannel * N_BINS + ix * N_Y + iy)[in_range]          # uncollapsed bin, subchannel-major, x slow (PROfit's global bin)
BASE_WEIGHT = scalars['non_genie_net_weight'][selected][in_range].astype(float)
Q2_TRUE = scalars['GTruth_gQ2'][selected][in_range].astype(float)
MA_WEIGHTS = zr._clean_ma_spline_weights(vectors['MaCCQE_UBGenie'][in_range])
STORED = {c: vectors[c][in_range] for c in VECTOR_COLUMNS if c != 'MaCCQE_UBGenie'}
del vectors, scalars
N_EVENTS = len(BASE_WEIGHT)
QUAD_MODEL = zr._prepare_quadratic_fa_splines(Q2_TRUE, MA_WEIGHTS)   # the production per-event quadratic in F_A


def bin_sums(event_weights):
    return np.bincount(U_IDX, weights=BASE_WEIGHT * event_weights, minlength=N_BINS_FULL)


CV_OCCUPANCY = bin_sums(np.ones(N_EVENTS))
ACTIVE = CV_OCCUPANCY > 0          # PROfit's ratios are 1 in empty bins; they carry no information and are excluded
print(f'Inside the analysis binning: {N_EVENTS:,} events (signal subchannel {np.sum(subchannel[in_range] == 0):,}, '
      f'background {np.sum(subchannel[in_range] == 1):,}); {ACTIVE.sum()} of {N_BINS_FULL} uncollapsed bins populated; '
      f'{np.sum(~QUAD_MODEL[3]):,} events with a fitted quadratic, {QUAD_MODEL[3].sum()} degenerate (Q2 = 0).')

## Do the stored branches match the production code?

Every seven-knot and CROSS weight is recomputed from `GTruth_gQ2` and `MaCCQE_UBGenie` through
`zexp_reweighting` and compared with what the file holds. The branches were written as float32,
so agreement at $\sim 10^{-7}$ is the expected floor; anything larger means the file and the code
disagree about a convention (this is what caught the dipole $F_A(0)$ mismatch of the 8 September
file). The same pass counts how many universe weights PROfit's guard would reset to 1.

In [ ]:
def prior_geometry(prior):
    shifts = zr.zexp_shift_matrix(prior.covariance, prior.use_pca)
    return dict(n_free=len(prior.free_a_values), shifts=shifts, pairs=zr.zexp_cross_pairs(len(prior.free_a_values)))


def exact_event_weights(prior, eta, geometry=None):
    # The production path: complete the coefficients at eta, evaluate F_A(Q2), then the per-event quadratic.
    geometry = geometry or prior_geometry(prior)
    a_full = zr.complete_zexp_a_values(prior.free_a_values + geometry['shifts'] @ np.asarray(eta, float), prior.kmax,
                                       prior.t0_gev2, t_cut_gev2=prior.t_cut_gev2, fa_q2_zero=prior.fa_q2_zero)
    return zr._weights_for_a_values(Q2_TRUE, MA_WEIGHTS, a_full, prior.t0_gev2, prior.t_cut_gev2, QUAD_MODEL)


def profit_guard(w):
    # PROcreate.cxx: a NaN/inf or > 30 universe weight is reset to 1 before it is filled.
    return np.where(np.isfinite(w) & (w <= WEIGHT_GUARD), w, 1.0)


GEOMETRY = {prior.name: prior_geometry(prior) for prior in PRIORS}
reproduction_rows = []
for prior in PRIORS:
    geo = GEOMETRY[prior.name]
    n = geo['n_free']
    worst, n_guard, n_weights = 0.0, 0, 0
    for j, branch in enumerate(prior.variation_branches):
        stored = STORED[branch]
        assert stored.shape == (N_EVENTS, len(KNOTS)), (branch, stored.shape)
        for k, knob in enumerate(KNOTS):
            eta = np.zeros(n)
            eta[j] = knob
            recomputed = exact_event_weights(prior, eta, geo)
            worst = max(worst, np.max(np.abs(stored[:, k] - recomputed) / np.maximum(np.abs(recomputed), 1e-12)))
        n_guard += np.sum(~(np.isfinite(stored) & (stored <= WEIGHT_GUARD)))
        n_weights += stored.size
    cross = STORED[prior.cross_branch]
    assert cross.shape == (N_EVENTS, 1 + len(geo['pairs'])), (prior.cross_branch, cross.shape)
    for p, (i, j) in enumerate(geo['pairs']):
        eta = np.zeros(n)
        eta[i] = eta[j] = 1.0
        recomputed = exact_event_weights(prior, eta, geo)
        worst = max(worst, np.max(np.abs(cross[:, 1 + p] - recomputed) / np.maximum(np.abs(recomputed), 1e-12)))
    recomputed0 = exact_event_weights(prior, np.zeros(n), geo)
    worst = max(worst, np.max(np.abs(cross[:, 0] - recomputed0) / np.maximum(np.abs(recomputed0), 1e-12)))
    n_guard += np.sum(~(np.isfinite(cross) & (cross <= WEIGHT_GUARD)))
    n_weights += cross.size
    reproduction_rows.append({'prior': prior.name, 'fit': FIT_KEY.get(prior.name, '-'), 'n_par': n,
                              'max |stored - recomputed| / recomputed': worst,
                              'universe weights': n_weights, 'guard resets (>30 or non-finite)': int(n_guard),
                              'max universe weight': float(np.max([np.max(STORED[b]) for b in (*prior.variation_branches, prior.cross_branch)]))})
REPRODUCTION = pd.DataFrame(reproduction_rows)
display(REPRODUCTION)
FLOAT32_FLOOR = REPRODUCTION['max |stored - recomputed| / recomputed'].max()
assert FLOAT32_FLOOR < 1e-5, 'the stored branches do not reproduce from the production code: stale file or changed convention'
print(f'Stored branches reproduce from the production code to {FLOAT32_FLOOR:.1e} (float32 storage floor); '
      f"{REPRODUCTION['guard resets (>30 or non-finite)'].sum()} universe weights would be reset by PROfit's guard.")

## PROfit's response, re-implemented from the stored branches

* `knot_ratios`: `PROcreate` fills one spectrum per knot with `weight_1 * weight_2 * w` (after the
  guard) and `PROsyst::FillSpline` takes the ratio to the CV, then, with `force_0_cv`, to the
  ratio at knot 0, which is the ratio to the knot-0 universe.
* `build_profit_splines` / `eval_profit_spline`: the CAFAna Hermite construction of `FillSpline`
  (quadratic end segments, cubic interior, slope $(y_{k+1}-y_{k-1})/2$) and the segment lookup of
  `GetSplineShift`, including its extrapolation beyond $\pm3$ (first / last segment continued).
* `cross_coefficients`: `FillSplineCrossQuad`, $e_{ij} = R(e_i+e_j) - s_i(1) - s_j(1) + 1$ with
  $R$ the CROSS universe $1+p$ over CROSS universe 0, pairs enumerated $a<b$ over the members
  in the XML `splines=` order, exactly as the C++ loop does.
* `additive_response` / `product_response`: `GetSplineFactor` with and without the group.

In [ ]:
N_SEGMENTS = len(KNOTS) - 1


def knot_ratios(stored_knots):
    # (n_events, 7) universe weights -> (7, n_bins) per-bin ratios to the knot-0 universe.
    sums = np.stack([bin_sums(profit_guard(stored_knots[:, k])) for k in range(len(KNOTS))])
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(sums[KNOT0] > 0, sums / sums[KNOT0], 1.0)


def build_profit_splines(ratios):
    # ratios: (7 knots, n_bins) -> coefficients (n_bins, 6 segments, 4); segment s starts at KNOTS[s],
    # polynomial in x = (eta - knot) / knot spacing. Transcribed from PROsyst::FillSpline.
    y = ratios
    n = y.shape[1]
    coeffs = np.zeros((n, N_SEGMENTS, 4))
    y1, y2, y3 = y[0], y[1], y[2]
    s = (y3 - y1) / 2
    coeffs[:, 0, :] = np.stack([y1, -2 * y1 + 2 * y2 - s, y1 - y2 + s, np.zeros(n)], axis=1)
    for i in range(1, len(KNOTS) - 2):
        y0, y1, y2, y3 = y[i - 1], y[i], y[i + 1], y[i + 2]
        m1, m2 = (y2 - y0) / 2, (y3 - y1) / 2
        coeffs[:, i, :] = np.stack([y1, m1, -3 * y1 + 3 * y2 - 2 * m1 - m2, 2 * y1 - 2 * y2 + m1 + m2], axis=1)
    y4, y5, y6 = y[-3], y[-2], y[-1]
    s = (y6 - y4) / 2
    coeffs[:, -1, :] = np.stack([y5, s, -y5 + y6 - s, np.zeros(n)], axis=1)
    return coeffs


def eval_profit_spline(coeffs, eta):
    # std::upper_bound on the segment knots, step back one: the last segment whose knot <= eta;
    # eta < -3 uses the first segment (x < 0), eta > 3 the last one (x > 1): extrapolation.
    segment = int(np.clip(np.searchsorted(KNOTS[:-1], eta, side='right') - 1, 0, N_SEGMENTS - 1))
    x = (eta - KNOTS[segment]) / (KNOTS[segment + 1] - KNOTS[segment])
    c = coeffs[:, segment, :]
    return c[:, 0] + x * (c[:, 1] + x * (c[:, 2] + x * c[:, 3]))


def cross_coefficients(prior, splines, members):
    # (n_bins, n_pairs) e_ij in the C++ pair order: a < b over `members` (the XML order).
    cross = STORED[prior.cross_branch]
    u0 = bin_sums(profit_guard(cross[:, 0]))
    n = len(members)
    e = np.zeros((N_BINS_FULL, n * (n - 1) // 2))
    p = 0
    for a in range(n):
        for b in range(a + 1, n):
            up = bin_sums(profit_guard(cross[:, 1 + p]))
            with np.errstate(invalid='ignore', divide='ignore'):
                ratio = np.where(u0 > 0, up / u0, 1.0)
            e[:, p] = ratio - eval_profit_spline(splines[members[a]], 1.0) - eval_profit_spline(splines[members[b]], 1.0) + 1.0
            p += 1
    return e


def build_profit_group(prior):
    xml_members, xml_file = XML_MEMBER_ORDER[prior.cross_branch]
    if set(xml_members) != set(prior.variation_branches):
        raise ValueError(f'{prior.cross_branch}: XML members {xml_members} are not the prior branches {prior.variation_branches}')
    splines = {branch: build_profit_splines(knot_ratios(STORED[branch])) for branch in prior.variation_branches}
    return dict(members=xml_members, xml_file=xml_file, splines=splines,
                cross=cross_coefficients(prior, splines, xml_members),
                xml_order_matches_branch=(xml_members == tuple(prior.variation_branches)))


def member_shifts(group, eta):
    # s_i(eta_i) per member in XML order, with eta indexed in branch (prior) order.
    return [eval_profit_spline(group['splines'][m], float(eta[group['branch_index'][m]])) for m in group['members']]


def additive_response(group, eta):
    s = member_shifts(group, eta)
    r = np.ones(N_BINS_FULL)
    n = len(s)
    p = 0
    for a in range(n):
        r += s[a] - 1.0
        for b in range(a + 1, n):
            r += group['cross'][:, p] * eta[group['branch_index'][group['members'][a]]] * eta[group['branch_index'][group['members'][b]]]
            p += 1
    return r


def product_response(group, eta):
    r = np.ones(N_BINS_FULL)
    for s in member_shifts(group, eta):
        r *= s
    return r


def exact_response(prior, eta, geo, cv_sums):
    return np.where(ACTIVE, bin_sums(exact_event_weights(prior, eta, geo)) / np.where(ACTIVE, cv_sums, 1.0), 1.0)


def on_axis_quadraticity(group):
    # FillSplineCrossQuad's run-time probe: fit 1 + b eta + d eta^2 through the +-1 knots and check
    # the +-2, +-3 knots against it; returns the worst relative departure over members and bins.
    worst = 0.0
    for m in group['members']:
        coeffs = group['splines'][m]
        sp, sm = eval_profit_spline(coeffs, 1.0), eval_profit_spline(coeffs, -1.0)
        b, d = 0.5 * (sp - sm), 0.5 * (sp + sm) - 1.0
        for knob in (-3.0, -2.0, 2.0, 3.0):
            got = eval_profit_spline(coeffs, knob)
            pred = 1.0 + b * knob + d * knob * knob
            worst = max(worst, np.max((np.abs(got - pred) / np.maximum(np.abs(got), 1e-3))[ACTIVE]))
    return worst


GROUPS = {}
for prior in PRIORS:
    group = build_profit_group(prior)
    group['branch_index'] = {branch: j for j, branch in enumerate(prior.variation_branches)}
    group['cv_sums'] = bin_sums(exact_event_weights(prior, np.zeros(GEOMETRY[prior.name]['n_free']), GEOMETRY[prior.name]))
    group['quadraticity'] = on_axis_quadraticity(group)
    GROUPS[prior.name] = group
    print(f"{prior.name:28s} members {group['members']} from {group['xml_file']}; XML order == branch order: "
          f"{group['xml_order_matches_branch']}; |e_ij| up to {np.abs(group['cross'][ACTIVE]).max():.3e}; "
          f"on-axis departure from a quadratic {group['quadraticity']:.1e}")

## Nodes and the comparison

Per prior: a regular grid on $[-3,3]^n$ (the knot range) plus random nodes there, and a coarser
grid plus random nodes on $[-7,7]^n$, where PROfit extrapolates and the uniform-prior posteriors
live. Random nodes sit between knots, where a Hermite segment that was only *pinned* at the
knots would still betray a wrong slope. Every node records the largest and median relative
per-bin difference of the additive and product responses from the exact one.

In [ ]:
def grid_nodes(n, half, step):
    axis = np.round(np.arange(-half, half + step / 2, step), 6)
    mesh = np.meshgrid(*([axis] * n), indexing='ij')
    return axis, np.column_stack([m.ravel() for m in mesh])


rng = np.random.default_rng(RNG_SEED)
NODE_SETS = {}
for prior in PRIORS:
    n = GEOMETRY[prior.name]['n_free']
    inner_axis, inner = grid_nodes(n, INNER_HALF, GRID_STEP[n])
    ext_axis, extended = grid_nodes(n, EXTENDED_HALF, EXTENDED_STEP[n])
    NODE_SETS[prior.name] = {
        'inner grid': (inner_axis, inner),
        'inner random': (None, rng.uniform(-INNER_HALF, INNER_HALF, (N_RANDOM, n))),
        'extended grid': (ext_axis, extended[np.max(np.abs(extended), axis=1) > INNER_HALF]),
        'extended random': (None, rng.uniform(-EXTENDED_HALF, EXTENDED_HALF, (N_RANDOM, n))),
    }


def rel_diff(model, exact):
    return (np.abs(model - exact) / np.abs(exact))[ACTIVE]


t0 = time.time()
node_rows = []
for prior in PRIORS:
    geo, group = GEOMETRY[prior.name], GROUPS[prior.name]
    for region, (_, nodes) in NODE_SETS[prior.name].items():
        for eta in nodes:
            exact = exact_response(prior, eta, geo, group['cv_sums'])
            d_add = rel_diff(additive_response(group, eta), exact)
            d_prod = rel_diff(product_response(group, eta), exact)
            node_rows.append({'prior': prior.name, 'region': region, 'radius': float(np.max(np.abs(eta))),
                              **{f'eta{k + 1}': float(eta[k]) for k in range(geo['n_free'])},
                              'additive_max': d_add.max(), 'additive_median': np.median(d_add),
                              'product_max': d_prod.max(), 'product_median': np.median(d_prod),
                              'exact_min': exact[ACTIVE].min(), 'exact_max': exact[ACTIVE].max()})
    print(f'{prior.name:28s} {sum(len(v[1]) for v in NODE_SETS[prior.name].values()):6d} nodes, {time.time() - t0:5.0f} s')
NODES = pd.DataFrame(node_rows)

## Verdict

In [ ]:
summary_rows = []
for prior in PRIORS:
    sub = NODES[NODES['prior'] == prior.name]
    inner = sub[sub['region'].str.startswith('inner')]
    extended = sub[sub['region'].str.startswith('extended')]
    rep = REPRODUCTION.set_index('prior').loc[prior.name]
    group = GROUPS[prior.name]
    row = {
        'prior': prior.name, 'fit': FIT_KEY.get(prior.name, '-'), 'n_par': GEOMETRY[prior.name]['n_free'],
        'nodes': len(sub),
        'additive |eta|<=3 max': inner['additive_max'].max(), 'additive |eta|<=3 median': inner['additive_median'].median(),
        'additive |eta|<=7 max': extended['additive_max'].max(),
        'product |eta|<=3 max': inner['product_max'].max(), 'product |eta|<=7 max': extended['product_max'].max(),
        'float32 floor': rep['max |stored - recomputed| / recomputed'],
        'guard resets': int(rep['guard resets (>30 or non-finite)']),
        'on-axis quadraticity': group['quadraticity'],
        'xml order == branch order': group['xml_order_matches_branch'],
    }
    row['pass'] = bool(row['additive |eta|<=3 max'] <= TOL_INNER and row['additive |eta|<=7 max'] <= TOL_EXTENDED
                       and row['xml order == branch order'])
    summary_rows.append(row)
SUMMARY = pd.DataFrame(summary_rows)
with pd.option_context('display.float_format', '{:.2e}'.format):
    display(SUMMARY.drop(columns=['nodes']))

worst_inner = SUMMARY['additive |eta|<=3 max'].max()
worst_ext = SUMMARY['additive |eta|<=7 max'].max()
print(f'Closure of the additive (spline_cross_quad) response: worst relative per-bin difference {worst_inner:.1e} on '
      f'[-3,3]^n (threshold {TOL_INNER:.0e}) and {worst_ext:.1e} on [-7,7]^n (threshold {TOL_EXTENDED:.0e}); '
      f'float32 storage floor {FLOAT32_FLOOR:.1e}.')
print(f"The multiplicative product is off by up to {SUMMARY['product |eta|<=3 max'].max():.1e} on [-3,3]^n "
      f"and {SUMMARY['product |eta|<=7 max'].max():.1e} on [-7,7]^n.")
if SUMMARY['pass'].all():
    print(f'PASS: all {len(SUMMARY)} priors close.')
else:
    print('FAIL:', ', '.join(SUMMARY.loc[~SUMMARY['pass'], 'prior']))
if SAVE_OUTPUTS:
    SUMMARY.to_csv(TABLE_DIR / 'cross_branch_per_bin_closure.csv', index=False)
    NODES.to_csv(TABLE_DIR / 'cross_branch_per_bin_closure_nodes.csv', index=False)
    print('Written:', TABLE_DIR / 'cross_branch_per_bin_closure.csv', 'and', TABLE_DIR / 'cross_branch_per_bin_closure_nodes.csv')

## Closure against distance from the central value

The envelope (largest per-bin difference over all nodes in a radius bin) of both responses,
one panel per prior. The additive response starts at the float32 floor and grows like $\eta^2$
because the rounding of the stored knot ratios propagates through the quadratic (it is a
precision effect of the stored branches, not a model error); the product's error is the missing
cross term and is orders of magnitude larger everywhere off the axes.

In [ ]:
RADIUS_EDGES = np.arange(0, EXTENDED_HALF + 0.5, 0.5)


def envelope(frame, column):
    idx = np.clip(np.digitize(frame['radius'], RADIUS_EDGES) - 1, 0, len(RADIUS_EDGES) - 2)
    return frame.groupby(idx)[column].max().reindex(range(len(RADIUS_EDGES) - 1))


ncol = 4
nrow = int(np.ceil((len(PRIORS) + 1) / ncol))        # one extra slot for the legend
with mpl.rc_context(PUBLICATION_RC):
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.8 * nrow), sharex=True, sharey=True, constrained_layout=True)
    flat = axes.ravel()
    for ax, prior in zip(flat, PRIORS):
        sub = NODES[NODES['prior'] == prior.name]
        for column, color in (('product_max', COLOR_PRODUCT), ('additive_max', COLOR_ADDITIVE)):
            ax.stairs(envelope(sub, column).to_numpy(), RADIUS_EDGES, color=color, lw=2, baseline=None)
        ax.axhline(FLOAT32_FLOOR, color='0.6', lw=1, ls=':')
        ax.axhline(TOL_INNER, color='0.3', lw=1, ls='--')
        ax.axvline(INNER_HALF, color='0.8', lw=0.8)
        ax.set_yscale('log')
        ax.set_title(f"{FIT_KEY.get(prior.name, prior.name)} ({GEOMETRY[prior.name]['n_free']} par)", fontsize=10)
        ax.grid(True, which='major', color='0.92', lw=0.6)
        if ax in axes[-1]:
            ax.set_xlabel(r'$\max_i |\eta_i|$')
    handles = [Line2D([], [], color=COLOR_PRODUCT, lw=2, label='product of splines\n(PROfit before)'),
               Line2D([], [], color=COLOR_ADDITIVE, lw=2, label='additive with cross terms\n(PROfit now)'),
               Line2D([], [], color='0.3', lw=1, ls='--', label=f'pass threshold on $[-3,3]^n$\n({TOL_INNER:.0e})'),
               Line2D([], [], color='0.6', lw=1, ls=':', label=f'float32 storage floor\n({FLOAT32_FLOOR:.0e})')]
    for ax in flat[len(PRIORS):]:
        ax.axis('off')
    flat[len(PRIORS)].legend(handles=handles, loc='center', frameon=False, fontsize=9, labelspacing=1.1)
    fig.supylabel('largest relative per-bin difference from the exact response', fontsize=10)
    if SAVE_OUTPUTS:
        fig.savefig(FIG_DIR / 'cross_branch_per_bin_closure_vs_radius.pdf', bbox_inches='tight', dpi=SAVE_DPI)
    plt.show()

## Maps for the two-parameter priors

Largest relative per-bin difference over the inner grid, additive (left) and product (right) on
a shared logarithmic colour scale. The product's error is the missing $\eta_1\eta_2$ term and
vanishes on the axes; the additive map should be featureless at the floor.

In [ ]:
two_par = [prior for prior in PRIORS if GEOMETRY[prior.name]['n_free'] == 2]
with mpl.rc_context(PUBLICATION_RC):
    fig, axes = plt.subplots(len(two_par), 2, figsize=(7.8, 3.3 * len(two_par)), constrained_layout=True, squeeze=False)
    vmin, vmax = max(FLOAT32_FLOOR / 10, 1e-9), max(NODES.loc[NODES['region'] == 'inner grid', 'product_max'].max(), 1e-6)
    norm = LogNorm(vmin=vmin, vmax=vmax)
    for row, prior in zip(axes, two_par):
        axis, _ = NODE_SETS[prior.name]['inner grid']
        step = axis[1] - axis[0]
        edges = np.r_[axis - step / 2, axis[-1] + step / 2]
        frame = NODES[(NODES['prior'] == prior.name) & (NODES['region'] == 'inner grid')]
        for ax, column, title in zip(row, ('additive_max', 'product_max'), ('additive with cross terms', 'product of splines')):
            values = frame.pivot(index='eta2', columns='eta1', values=column).reindex(index=axis, columns=axis).to_numpy()
            mesh = ax.pcolormesh(edges, edges, np.clip(values, vmin, None), cmap='viridis', norm=norm, rasterized=True)
            ax.set_aspect('equal')
            ax.set_xlabel(r'$\eta_1$')
            ax.set_ylabel(r'$\eta_2$')
            ax.set_title(f"{FIT_KEY.get(prior.name, prior.name)}: {title}", fontsize=10)
    cbar = fig.colorbar(mesh, ax=axes, pad=0.02, shrink=0.6)
    cbar.set_label('largest relative per-bin difference')
    if SAVE_OUTPUTS:
        fig.savefig(FIG_DIR / 'cross_branch_per_bin_closure_maps.pdf', bbox_inches='tight', dpi=SAVE_DPI)
    plt.show()